# Выбор стратегий предобработки

Здесь систематически подбираем оси предобработки для всех моделей на двух наборах признаков (significant 10 и no_collinear 25):

1. импутация пропусков (median/mode, KNN, MICE, ручная клиническая);
2. балансировка классов (none, class_weight, SMOTE);
3. кодирование категориальных (one-hot, target encoding);
4. преобразования количественных признаков (без, ручная схема логарифм + Йео-Джонсон, Йео-Джонсон ко всем).

Порядок исследования: сначала по очереди смотрим каждую ось при фиксированных остальных (контролируемое сравнение), затем подбираем три основные оси одновременно, затем сводим в таблицу и делаем вывод.

Ручная импутация и преобразования признаков - попытки заменить простую автоматическую предобработку клинически осмысленной. Проверяем, дают ли они прирост разделяющей способности. Если прироста нет, оставляем простую связку (KNN, one-hot, без преобразований), а сложные варианты держим в резерве: на других данных прирост может появиться, и тогда выбор придется пересмотреть.

Каркас без утечки (`src/modeling.py`): train/test split один раз, вся предобработка обучается внутри фолдов 5-фолдовой стратифицированной CV, отложенный тест не трогаем. Гиперпараметры здесь по умолчанию - это сравнение стратегий, а не финальная оценка (она в ноутбуках 07 тюнинг и 08 тест). Метрика - ROC-AUC по фолдам, она не зависит от порога, поэтому корректно сравнивает стратегии.

Технически: совместную сетку основных комбинаций (импутация, балансировка, кодирование) считаем один раз, а исследования отдельных осей показываем как ее контролируемые срезы. Преобразования признаков считаем отдельным блоком, так как они меняют числовой конвейер.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import pandas as pd
from IPython.display import display
from src import modeling, config
pd.set_option('display.max_rows', 300)

# Оси перебора.
MODELS = ['logreg', 'rf', 'xgb', 'lgbm', 'catboost', 'svm', 'knn']
SETS = ['significant', 'no_collinear']
IMPUTE = ['median_mode', 'knn', 'mice', 'manual']
BALANCE = ['none', 'class_weight', 'smote']
ENCODE = ['onehot', 'target']

# Совместная сетка всех валидных комбинаций (считается один раз, несколько минут).
# CatBoost обрабатывает категориальные нативно (без one-hot), поэтому у него
# кодирование одно - native. Несовместимые сочетания отсекаются внутри run_grid.
configs = []
for m in MODELS:
    encs = ['native'] if m == 'catboost' else ENCODE
    for imp in IMPUTE:
        for bal in BALANCE:
            for enc in encs:
                for fs in SETS:
                    configs.append((m, imp, bal, fs, enc))
joint = modeling.run_grid(configs)
joint.to_csv(config.TABLES_DIR / 'strategy_grid.csv', index=False,
             encoding='utf-8-sig')
print('\nВалидных конфигураций:', len(joint))

# Базовое кодирование для срезов: one-hot, а у CatBoost - его native.
BASE_ENC = ['onehot', 'native']


def winners(dft, axis):
    """Лучшее значение оси axis для каждой модели и набора по ROC-AUC."""
    idx = dft.groupby(['модель', 'набор'])['roc_auc'].idxmax()
    cols = ['модель', 'набор', axis, 'roc_auc', 'roc_auc_sd']
    return dft.loc[idx, cols].sort_values(['набор', 'roc_auc'],
                                          ascending=[True, False])

## 1. Импутация (у всех моделей, на двух наборах)

Контролируемый срез совместной сетки: балансировка none, кодирование one-hot (у CatBoost - его native), меняется только импутация (median/mode, KNN, MICE, ручная клиническая). ROC-AUC не зависит от порога, поэтому фиксированная балансировка тут не мешает сравнению. Подробное описание ручной стратегии - сразу после таблицы.

In [ ]:
s1 = joint[(joint['балансировка'] == 'none') & (joint['кодирование'].isin(BASE_ENC))]
for fs in SETS:
    print('Набор', fs)
    display(s1[s1['набор'] == fs].pivot_table(index='модель', columns='импутация',
                                              values='roc_auc').round(3))
print('Лучшая импутация для каждой модели:')
winners(s1, 'импутация')

### Ручная клиническая импутация: что это и как устроено

Автоматические стратегии (медиана, KNN, MICE) заполняют все колонки одним механизмом. Ручная импутация заполняет каждый показатель по его клиническому смыслу. Реализована трансформером `ManualImputer` (`src/manual_impute.py`), он встроен в пайплайн первым шагом и обучается только на train внутри фолдов, поэтому утечки нет. Логика делится на три части.

Детерминированные связи между колонками. Они вытекают из определений, не зависят от данных и потому не дают утечки в принципе:
- возраст манифестации СД = возраст минус длительность СД (и обратные перестановки: недостающее из двух известных);
- ЛПНП по формуле Фридвальда: общий холестерин минус ЛПВП минус ТГ/2.2, применяем при ТГ ниже 4.5 ммоль/л и только если результат положительный;
- степень тяжести ДКА из pH по порогам ADA: pH не ниже 7.25 - легкая, 7.0-7.24 - средняя, ниже 7.0 - тяжелая.

Статистики с клинической стратификацией. Обучаются на train (медианы внутри групп), поэтому встроены в fit/transform:
- креатинин заполняем медианой по полу, так как референсные значения зависят от пола;
- острые и связанные с тяжестью показатели (HbA1c, pH, BE, лактат, калий, натрий) заполняем медианой внутри группы по степени тяжести ДКА. Решение врача: чем тяжелее эпизод, тем хуже электролиты, ацидоз и хронический контроль, поэтому групповая медиана точнее общей;
- остальные количественные - общей медианой train (клинической стратификации для них нет).

Категориальные показатели. Здесь не везде уместна мода, часть пропусков информативна:
- ретинопатия, ХБП-С, ХБП-А - отдельным уровнем "не обследовано": пропуск означает, что осмотр или оценку не проводили, это сам по себе сигнал, а не норма (ХБП-А самая дырявая категория, около 48% пропусков);
- невролог - клиническим нулем "нет осложнения": мода тут равна 1 и увела бы пропуски в сторону "есть";
- прочие категориальные (тип СД, пол, вид инсулинотерапии, алкоголь) - модой train.

Зачем мы это проверяем. Гипотеза в том, что клинически осмысленное заполнение точнее общей статистики и поднимет качество. Ниже ручная стратегия участвует в сравнении импутаций (столбец manual в таблице выше) на равных с автоматическими, тем же leak-free механизмом. Если ROC-AUC ручной импутации не выше KNN, клинические правила не дают прироста разделяющей способности, и мы остаемся на KNN. Таблица правил с долями пропусков по колонкам - ниже.

In [ ]:
from src import manual_impute

# Таблица правил ручной импутации: колонка, доля пропусков, правило, обоснование.
display(manual_impute.decisions_table())

## 2. Балансировка классов (у всех моделей, на двух наборах)

Срез: импутация KNN, кодирование one-hot (у CatBoost native), меняется балансировка (none, class_weight, SMOTE). У kNN-модели class_weight не поддерживается, у CatBoost с нативными категориями нет SMOTE - в этих клетках прочерк.

In [ ]:
s2 = joint[(joint['импутация'] == 'knn') & (joint['кодирование'].isin(BASE_ENC))]
for fs in SETS:
    print('Набор', fs)
    display(s2[s2['набор'] == fs].pivot_table(index='модель', columns='балансировка',
                                              values='roc_auc').round(3))
print('Лучшая балансировка для каждой модели:')
winners(s2, 'балансировка')

## 3. Кодирование категориальных (у всех моделей, на двух наборах)

Срез: импутация KNN, балансировка none, меняется кодирование. one-hot создает по столбцу на уровень, target encoding заменяет категорию средней частотой исхода в ней (с внутренней перекрестной подгонкой, без утечки). CatBoost кодирует категории сам (native), поэтому у него отдельный столбец native, а не one-hot или target.

In [ ]:
s3 = joint[(joint['импутация'] == 'knn') & (joint['балансировка'] == 'none')]
for fs in SETS:
    print('Набор', fs)
    display(s3[s3['набор'] == fs].pivot_table(index='модель', columns='кодирование',
                                              values='roc_auc').round(3))
print('Лучшее кодирование для каждой модели:')
winners(s3, 'кодирование')

## 4. Преобразования количественных признаков

Скошенные количественные показатели (липиды, глюкоза, мочевина, креатинин, лактат, HbA1c) можно нормализовать, это иногда помогает линейным моделям и методам, основанным на расстоянии. Проверяем три варианта числового конвейера:

- none: только стандартизация;
- manual: логарифм для скошенных вправо положительных показателей (ТГ, общий холестерин, лактат, HbA1c, ЛПНП, ЛПВП, глюкоза, мочевина, креатинин) и Йео-Джонсон для BE и длительности СД, затем стандартизация;
- yeojohnson_all: степенное преобразование Йео-Джонсона ко всем количественным, затем стандартизация.

Преобразования и импутация обучаются на train внутри фолдов, оценка - по out-of-fold ROC-AUC с бутстрэп-ДИ. Деревья (rf, lgbm, catboost) инвариантны к монотонным преобразованиям отдельных признаков и служат контролем: у них варианты должны совпасть с точностью до случайности фолдов. Масштабочувствительные модели (logreg, svm, knn) - те, у кого преобразования могли бы дать эффект.

In [ ]:
from src import transforms

# Отдельный блок: преобразования числовых обучаются на train внутри фолдов,
# оценка по out-of-fold ROC-AUC с бутстрэп-ДИ (src/transforms.py). Импутация KNN,
# взвешивание классов. Деревья включены, чтобы показать их инвариантность.
tr = transforms.run()
for fs in transforms.FSETS:
    print('Набор', fs)
    display(tr[tr['набор'] == fs].pivot_table(
        index='модель', columns='преобразование', values='ROC_AUC').round(3))

## 5. Совместный подбор всех трех стратегий

Лучшая комбинация импутации, балансировки и кодирования для каждой модели и набора по ROC-AUC. Интересно сравнить, совпадут ли победители с раздельными исследованиями 1-3.

In [ ]:
cols = ['модель', 'набор', 'импутация', 'балансировка', 'кодирование',
        'roc_auc', 'roc_auc_sd', 'sens', 'spec', 'brier']
idx = joint.groupby(['модель', 'набор'])['roc_auc'].idxmax()
best = joint.loc[idx, cols].sort_values(['набор', 'roc_auc'], ascending=[True, False])
display(best)

## 6. Сводная таблица и вывод

Топ-конфигурации по всей сетке и итог по выбору стратегий.

In [ ]:
print('Топ-12 конфигураций по ROC-AUC во всей сетке:')
top = joint.sort_values('roc_auc', ascending=False).head(12)
display(top[['модель', 'импутация', 'балансировка', 'кодирование', 'набор',
             'roc_auc', 'roc_auc_sd', 'sens', 'spec', 'brier']])

# Сколько раз каждая стратегия выигрывала в совместном подборе (по моделям и наборам).
print('\nЧастота победившей стратегии в совместном подборе:')
for axis in ['импутация', 'балансировка', 'кодирование']:
    print(axis, dict(best[axis].value_counts()))

### Вывод

1. Импутация. KNN - рабочая стратегия. На наборе no_collinear она строго лучшая у всех семи моделей (rf 0.760, catboost 0.747, lgbm 0.724, logreg 0.712, xgb 0.707), автоматические median/mice и ручная клиническая там заметно отстают (например, ручная для rf 0.709 против 0.760, для lgbm 0.643 против 0.724). На слабом наборе significant ручная импутация местами выходит чуть вперед (catboost 0.737 против knn 0.721, logreg 0.728 против 0.719), но это набор из 10 признаков, который мы не берем основным, и разрыв лежит внутри стандартного отклонения по фолдам. В совместном подборе KNN выигрывает в 7 парах модель-набор из 14, ручная в 4 (все на significant), MICE в 3, median/mode ни разу. Клинические правила не дают прироста разделяющей способности там, где он важен, поэтому остаемся на KNN. Если на других данных ручная импутация даст устойчивый прирост, выбор нужно будет пересмотреть.
2. Балансировка. На ROC-AUC влияет слабо: none и class_weight идут вровень (6 и 5 побед из 14), SMOTE 3. Это ожидаемо, AUC не зависит от порога, а балансировка по сути сдвигает порог. Балансировку оставляем как настраиваемую ручку для баланса чувствительность-специфичность, сам порог подбираем отдельно (ноутбук 09). Подробное честное исследование синтетической балансировки (SMOTENC, BorderlineSMOTE, ADASYN) - ноутбук 12.
3. Кодирование. Для линейных моделей и SVM лучше one-hot (logreg 0.712 против target 0.677 на no_collinear, SVM 0.677 против 0.643), для деревьев one-hot и target вровень, у kNN-модели чуть лучше target. CatBoost кодирует категории сам (native, упорядоченные target-статистики без утечки) и на no_collinear дает 0.747. Берем: для всех моделей one-hot, для CatBoost - его нативную обработку.
4. Преобразования количественных. Прироста не дают. Деревья инвариантны к монотонным преобразованиям, что и подтвердилось: rf на no_collinear 0.773 (без), 0.756 (ручная схема), 0.775 (Йео-Джонсон ко всем), CatBoost 0.747 / 0.742 / 0.746 - различия в пределах случайности фолдов. У масштабочувствительных моделей (logreg, svm, knn) преобразования двигают ROC-AUC на ±0.01 в обе стороны, без устойчивого выигрыша, доверительные интервалы полностью перекрываются (logreg significant 0.728 / 0.732 / 0.734). Преобразования признаков не выбираем. Как и с ручной импутацией: появись устойчивый прирост на других данных - это повод вернуться к вопросу.
5. Совместный подбор не дал комбинации выше общего потолка: лучшее - случайный лес с KNN, class_weight, one-hot на no_collinear (ROC-AUC 0.780), затем CatBoost с KNN, class_weight, native (0.763). Это оптимистичная одиночная CV с дефолтными гиперпараметрами, она систематически завышает деревья. Честные числа дают вложенная CV (ноутбук 07) и отложенный тест (ноутбук 08).
6. Рабочая связка для дальнейших этапов: KNN-импутация, one-hot (CatBoost - native), без ручной импутации и без преобразований признаков, балансировка как настраиваемая ручка. На этой основе в ноутбуке 07 подобраны гиперпараметры, в 08 проверено на тесте.